# Genre EDA — listen-wiseer

Covers:
1. Genre taxonomy coverage
2. Audio profiles by gen_4 (violin + radar)
3. Genre pairplot by gen_4
4. Genre pairplot by gen_8
5. Genre clustering (dendrogram, PCA, elbow, silhouette, TSNE)

In [ ]:
import sys
from pathlib import Path

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))

import warnings

warnings.filterwarnings("ignore")
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 110
DB = str(ROOT / "data/listen_wiseer.db")
con = duckdb.connect(DB, read_only=True)

In [ ]:
tp = con.execute("SELECT * FROM track_profile").df()
print(f"Loaded {len(tp):,} tracks")

## 1. Genre taxonomy coverage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# gen_4 breakdown
gen4 = tp["gen_4"].value_counts().head(20)
gen4.plot.barh(ax=axes[0])
axes[0].set_title("Top gen_4 categories (track count)")
axes[0].invert_yaxis()

# my_genre breakdown
my_g = tp["my_genre"].value_counts().head(20)
my_g.plot.barh(ax=axes[1])
axes[1].set_title("Top my_genre categories (track count)")
axes[1].invert_yaxis()

plt.tight_layout()

In [ ]:
# top first_genres (raw Spotify genre labels)
tp["first_genre"].value_counts().head(30).to_frame("count")

In [ ]:
# genre_cat (coarser tag)
tp["genre_cat"].value_counts(dropna=False)

In [ ]:
# ENOA area plot: genre groups by top/left coordinates
# top and left are ENOA map coordinates from the genre_map table


def plot_enoa_area(genre_groups, title="ENOA genre map"):
    """Draw rectangles for each genre using min/max top and left coordinates."""
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = sns.color_palette("muted", n_colors=len(genre_groups))
    for (genre, row), color in zip(genre_groups.iterrows(), colors, strict=False):
        left_min = row.get("left_min", row.get(("left", "min"), 0))
        left_max = row.get("left_max", row.get(("left", "max"), 0))
        top_min = row.get("top_min", row.get(("top", "min"), 0))
        top_max = row.get("top_max", row.get(("top", "max"), 0))
        width = left_max - left_min
        height = top_max - top_min
        rect = plt.Rectangle(
            (left_min, top_min),
            width,
            height,
            linewidth=1.5,
            edgecolor=color,
            facecolor=(*color[:3], 0.15),
            label=genre,
        )
        ax.add_patch(rect)
        cx = left_min + width / 2
        cy = top_min + height / 2
        ax.text(
            cx,
            cy,
            str(genre),
            ha="center",
            va="center",
            fontsize=8,
            color=color if hasattr(color, "__iter__") else "black",
        )
    ax.autoscale()
    ax.set_xlabel("left (ENOA x)")
    ax.set_ylabel("top (ENOA y)")
    ax.set_title(title)
    plt.tight_layout()


# gen_4 ENOA area
tp_enoa = tp.dropna(subset=["top", "left", "gen_4"])
genre_groups_4 = tp_enoa.groupby("gen_4")[["top", "left"]].agg(["min", "max"])
genre_groups_4.columns = ["_".join(c) for c in genre_groups_4.columns]
plot_enoa_area(genre_groups_4, title="ENOA area by gen_4")

In [ ]:
# gen_8 ENOA area
tp_enoa8 = tp.dropna(subset=["top", "left", "gen_8"])
genre_groups_8 = tp_enoa8.groupby("gen_8")[["top", "left"]].agg(["min", "max"])
genre_groups_8.columns = ["_".join(c) for c in genre_groups_8.columns]
plot_enoa_area(genre_groups_8, title="ENOA area by gen_8")

## 2. Audio profiles by gen_4

In [ ]:
viz_features = ["danceability", "energy", "valence", "acousticness", "instrumentalness"]
tp_clean = tp.dropna(subset=["gen_4"])

fig, axes = plt.subplots(1, len(viz_features), figsize=(16, 5))
for ax, feat in zip(axes, viz_features, strict=False):
    order = tp_clean.groupby("gen_4")[feat].median().sort_values().index
    sns.violinplot(data=tp_clean, x="gen_4", y=feat, order=order, ax=ax, inner="box", cut=0)
    ax.set_xlabel("")
    ax.set_title(feat)
    ax.tick_params(axis="x", rotation=30)
fig.suptitle("Audio profiles by gen_4", fontsize=14, y=1.01)
plt.tight_layout()

In [ ]:
radar_feats = [
    "danceability",
    "energy",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
]
gen4_medians = tp_clean.groupby("gen_4")[radar_feats].median()

N = len(radar_feats)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, axes = plt.subplots(
    1, len(gen4_medians), figsize=(4 * len(gen4_medians), 4), subplot_kw=dict(polar=True)
)
if len(gen4_medians) == 1:
    axes = [axes]

for ax, (genre, row) in zip(axes, gen4_medians.iterrows(), strict=False):
    vals = row.tolist() + [row.tolist()[0]]
    ax.plot(angles, vals, linewidth=2)
    ax.fill(angles, vals, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_feats, size=8)
    ax.set_ylim(0, 1)
    ax.set_title(genre, pad=15)

fig.suptitle("Median audio feature radar by gen_4", y=1.05)
plt.tight_layout()

## 3. Genre pairplot by gen_4

In [ ]:
pairplot_cols = [
    "danceability",
    "energy",
    "valence",
    "acousticness",
    "instrumentalness",
    "tempo",
    "gen_4",
]
pp_df = tp[pairplot_cols].dropna(subset=["gen_4"])
g = sns.pairplot(pp_df, hue="gen_4", plot_kws={"alpha": 0.3, "s": 10})
g.fig.suptitle("Pairplot by gen_4", y=1.01)
plt.tight_layout()

## 4. Genre pairplot by gen_8

In [ ]:
pairplot_cols_8 = [
    "danceability",
    "energy",
    "valence",
    "acousticness",
    "instrumentalness",
    "tempo",
    "gen_8",
]
pp_df8 = tp[pairplot_cols_8].dropna(subset=["gen_8"])
g8 = sns.pairplot(pp_df8, hue="gen_8", plot_kws={"alpha": 0.3, "s": 10})
g8.fig.suptitle("Pairplot by gen_8", y=1.01)
plt.tight_layout()

## 5. Genre clustering

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import calinski_harabasz_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

In [ ]:
audio_cols = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "popularity",
]

tp_cluster = con.execute("SELECT * FROM track_profile").df()
tp_cluster = tp_cluster.dropna(subset=audio_cols).reset_index(drop=True)
X = tp_cluster[audio_cols]
print(f"Clustering on {len(X):,} tracks x {len(audio_cols)} features")

In [ ]:
def plot_genres_dendrogram(df, audio_cols=audio_cols):
    """Hierarchical clustering dendrogram on genre centroids."""
    genre_centroids = df.dropna(subset=["gen_4"] + audio_cols).groupby("gen_4")[audio_cols].mean()
    scaler = MinMaxScaler()
    Z = linkage(scaler.fit_transform(genre_centroids), method="ward")
    fig, ax = plt.subplots(figsize=(10, 5))
    dendrogram(Z, labels=genre_centroids.index.tolist(), ax=ax)
    ax.set_title("Genre dendrogram (ward linkage on scaled audio centroids)")
    plt.tight_layout()


plot_genres_dendrogram(tp_cluster)

In [ ]:
def select_pca_n_components(X, threshold=0.90):
    """Plot cumulative explained variance; return n where cumvar >= threshold."""
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    pca = PCA().fit(X_scaled)
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    n = int(np.searchsorted(cumvar, threshold)) + 1

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(range(1, len(cumvar) + 1), cumvar, marker="o")
    ax.axhline(threshold, color="red", linestyle="--", label=f"{threshold:.0%} threshold")
    ax.axvline(n, color="orange", linestyle="--", label=f"n={n}")
    ax.set_xlabel("n_components")
    ax.set_ylabel("Cumulative explained variance")
    ax.set_title("PCA explained variance")
    ax.legend()
    plt.tight_layout()
    return n


n_components = select_pca_n_components(X)
print(f"Selected n_components: {n_components}")

In [ ]:
def plot_elbow_method(X, k_range=range(2, 12)):
    """Inertia vs k elbow plot; return k at elbow (min second derivative)."""
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    inertias = [
        KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled).inertia_ for k in k_range
    ]

    diffs2 = np.diff(np.diff(inertias))
    k_elbow = list(k_range)[np.argmax(diffs2) + 1]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(list(k_range), inertias, marker="o")
    ax.axvline(k_elbow, color="red", linestyle="--", label=f"elbow k={k_elbow}")
    ax.set_xlabel("k")
    ax.set_ylabel("Inertia")
    ax.set_title("Elbow method")
    ax.legend()
    plt.tight_layout()
    return k_elbow


n_clusters = plot_elbow_method(X)
print(f"Selected n_clusters: {n_clusters}")

In [ ]:
def plot_silhouette_scores(X, k_range=range(2, 12)):
    """Silhouette score vs k."""
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    scores = [
        silhouette_score(
            X_scaled, KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_scaled)
        )
        for k in k_range
    ]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(list(k_range), scores, marker="o")
    ax.set_xlabel("k")
    ax.set_ylabel("Silhouette score")
    ax.set_title("Silhouette scores")
    plt.tight_layout()


plot_silhouette_scores(X)

In [ ]:
def plot_tsne(X, n_components, labels, model_name):
    """2-D TSNE scatter colored by cluster label."""
    scaler = MinMaxScaler()
    pca = PCA(n_components=n_components)
    X_scaled = scaler.fit_transform(X)
    X_pca = pca.fit_transform(X_scaled)
    X_tsne = TSNE(n_components=2, random_state=42).fit_transform(X_pca)

    fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels, cmap="tab10", alpha=0.5, s=10)
    ax.set_title(f"TSNE — {model_name}")
    plt.colorbar(scatter, ax=ax, label="cluster")
    plt.tight_layout()

In [ ]:
def plot_cluster_features(df, model_name, audio_cols=audio_cols):
    """Boxplot of audio features per cluster."""
    n_features = len(audio_cols)
    fig, axes = plt.subplots(2, (n_features + 1) // 2, figsize=(16, 8))
    for ax, col in zip(axes.flatten(), audio_cols, strict=False):
        sns.boxplot(data=df, x=model_name, y=col, ax=ax)
        ax.set_title(col)
        ax.set_xlabel("cluster")
    fig.suptitle(f"Audio features per cluster — {model_name}", fontsize=13, y=1.01)
    plt.tight_layout()

In [ ]:
def config_model_pipeline(n_components, n_clusters):
    return [
        Pipeline(
            [
                ("scaler", MinMaxScaler()),
                ("pca", PCA(n_components=n_components)),
                ("classifier", KMeans(n_clusters=n_clusters, n_init=10, random_state=42)),
            ]
        ),
        Pipeline(
            [
                ("scaler", MinMaxScaler()),
                ("pca", PCA(n_components=n_components)),
                ("classifier", SpectralClustering(n_clusters=n_clusters, random_state=42)),
            ]
        ),
    ]


pipelines = config_model_pipeline(n_components, n_clusters)

for model in pipelines:
    model_name = type(model.named_steps["classifier"]).__name__
    labels = model.fit_predict(X).tolist()
    tp_cluster[model_name] = labels

    silhouette_avg = silhouette_score(X, labels)
    print(f"{model_name} Silhouette Score: {silhouette_avg:.4f}")
    ch = calinski_harabasz_score(X, labels)
    print(f"{model_name} Calinski-Harabasz Index: {ch:.2f}")

    plot_tsne(X, n_components, labels, model_name)
    plot_cluster_features(tp_cluster, model_name)

In [ ]:
tp_cluster["KMeans"].value_counts().plot(kind="bar", title="KMeans cluster sizes")
plt.tight_layout()

In [ ]:
tp_cluster[tp_cluster["KMeans"] == 0][["track_name", "gen_4", "first_genre"]].head(20)

In [ ]:
tp_cluster["SpectralClustering"].value_counts().plot(
    kind="bar", title="SpectralClustering cluster sizes"
)
plt.tight_layout()

In [ ]:
tp_cluster[tp_cluster["SpectralClustering"] == 0][["track_name", "gen_4", "first_genre"]].head(20)

In [ ]:
con.close()
print("Done.")